1. 共用參數設定


In [ ]:
'''
Measure the elapsed time of every code cell, including cells that raise errors.
記錄每個 Code Cell 的執行時間，即使發生錯誤也顯示已耗費的時間。
'''
from time import perf_counter as _cell_perf_counter
from IPython import get_ipython as _cell_get_ipython

_CELL_TIMER_SHELL = _cell_get_ipython()
if _CELL_TIMER_SHELL is None:
    raise RuntimeError('請在 Jupyter Notebook 中執行本教材。')
_CELL_TIMER_STATE = {'started': _cell_perf_counter()}

def _cell_timer_begin(_info):
    _CELL_TIMER_STATE['started'] = _cell_perf_counter()

def _cell_timer_end(result):
    started = _CELL_TIMER_STATE.pop('started', None)
    if started is None:
        return
    minutes, seconds = divmod(_cell_perf_counter() - started, 60)
    elapsed = f'{int(minutes)} 分 {seconds:05.2f} 秒' if minutes else f'{seconds:.2f} 秒'
    status = '執行失敗' if result.error_before_exec or result.error_in_exec else '執行完成'
    print(f'⏱ 本格{status}，耗時：{elapsed}')

'''
Replace existing timers when the first cell is executed again.
重新執行第一格時，先取消舊計時器，避免重複顯示耗時。
'''
for _event_name, _callback in globals().get('_CELL_TIMER_CALLBACKS', ()):
    try:
        _CELL_TIMER_SHELL.events.unregister(_event_name, _callback)
    except ValueError:
        pass
_CELL_TIMER_CALLBACKS = (
    ('pre_run_cell', _cell_timer_begin),
    ('post_run_cell', _cell_timer_end),
)
for _event_name, _callback in _CELL_TIMER_CALLBACKS:
    _CELL_TIMER_SHELL.events.register(_event_name, _callback)

'''
Import standard-library tools for editable paths and local service settings.
匯入設定路徑與本機服務參數所需的 Python 內建工具。
'''
import os
from pathlib import Path
ROOT = Path('/kaggle/working') if Path('/kaggle/working').is_dir() else Path.home() / 'ai_course'
ROOT = ROOT.expanduser().resolve()  # Ubuntu 可改成 Path('/home/your_user/ai_course')
SCRATCH_ROOT = Path('/tmp') if Path('/kaggle/working').is_dir() else ROOT / 'cache'
SCRATCH_ROOT = SCRATCH_ROOT.expanduser().resolve()  # 可改成其他儲存裝置或模型磁碟
ROOT.mkdir(parents=True, exist_ok=True)
SCRATCH_ROOT.mkdir(parents=True, exist_ok=True)

COMFY = ROOT / 'ComfyUI-ZImageTurbo'
GPU_COUNT = 2  # Kaggle T4 ×2；可依 NVIDIA 主機調整
MODEL_GPU_ID = 0  # 圖片生成模型
AUX_GPU_ID = 1  # 文字編碼器與 VAE

'''
Select the pinned ComfyUI release; update only after verifying custom nodes.
選擇固定的 ComfyUI 版本，更新前須先驗證自訂節點相容性。
'''
COMFY_REF = 'v0.45.0'
COMFY_PORT = 8189  # 8189、8190 等未占用連接埠
HF_CACHE_DIR = SCRATCH_ROOT / 'zimage_hf'

'''
Apply the cache path before model downloads begin.
模型下載之前先套用設定的快取路徑。
'''
os.environ['HF_HOME'] = str(HF_CACHE_DIR)
os.environ['HF_HUB_CACHE'] = str(HF_CACHE_DIR / 'hub')
REMOTE_PORT = 8765  # 例如 8765、8766，範圍 1024～65535
REMOTE_SERVICE_ID = '06_Z-Image-Turbo_圖片生成_Kaggle'

'''
Place optional API job data inside the configured workspace.
將遠端工作與暫存資料存放在第一格設定的工作空間底下。
'''
REMOTE_ROOT = ROOT / 'remote_api_jobs' / REMOTE_SERVICE_ID
REMOTE_QUEUE_CAPACITY = 32  # 建議 12～32，依課堂人數調整
REMOTE_UPLOAD_CONCURRENCY = 2  # 建議 1～2

'''
Choose output retention and storage safety limits.
設定遠端成果保留期限、空間限制及主機磁碟最低剩餘量。
'''
REMOTE_RESULT_TTL_HOURS = 24
REMOTE_RESULT_LIMIT_GIB = 8
REMOTE_MIN_FREE_GIB = 2
REMOTE_STALL_HOURS = float('inf')  # 不限制單次生成時長，也不將執行中的工作誤判為停滯
assert 1024 <= REMOTE_PORT <= 65535, 'REMOTE_PORT 必須介於 1024～65535。'
assert 1 <= REMOTE_QUEUE_CAPACITY <= 1024, 'FIFO 佇列容量設定無效。'
assert 1 <= REMOTE_UPLOAD_CONCURRENCY <= 8, '上傳並發限制設定無效。'
assert REMOTE_RESULT_TTL_HOURS > 0 and REMOTE_RESULT_LIMIT_GIB > 0 and REMOTE_MIN_FREE_GIB >= 0
assert REMOTE_STALL_HOURS > 0

'''
Avoid a local ComfyUI port collision with the optional FIFO API.
避免 ComfyUI 與選用 FIFO API 使用同一個本機 Port。
'''
assert 1024 <= COMFY_PORT <= 65535 and COMFY_PORT != REMOTE_PORT
print('✓ 第 1 節完成：共用參數已設定，請繼續執行環境安裝。')

assert 1 <= GPU_COUNT <= 4, 'GPU_COUNT 必須介於 1～4。'
assert all(0 <= gpu < GPU_COUNT for gpu in (MODEL_GPU_ID, AUX_GPU_ID)), 'GPU ID 超出 GPU_COUNT 的範圍。'
assert GPU_COUNT != 1 or MODEL_GPU_ID == AUX_GPU_ID, '單張 GPU 時，兩種元件應使用相同的 GPU ID。'


2. 安裝套件


In [ ]:
'''
Import the tools for installing ComfyUI.
匯入安裝 ComfyUI 所需工具。
'''
import subprocess
import sys
from pathlib import Path

REPOS = [
    ('https://github.com/Comfy-Org/ComfyUI.git', COMFY),
    ('https://github.com/city96/ComfyUI-GGUF.git', COMFY / 'custom_nodes/ComfyUI-GGUF'),
    ('https://github.com/pollockjj/ComfyUI-MultiGPU.git', COMFY / 'custom_nodes/ComfyUI-MultiGPU'),
]
old_commits = {}
for url, folder in REPOS:
    if folder.exists() and not (folder / '.git').exists():
        raise RuntimeError(f'不是 Git 倉庫：{folder}')
    if (folder / '.git').exists():
        old_commits[folder] = subprocess.check_output(['git', '-C', str(folder), 'rev-parse', 'HEAD'], text=True).strip()
    else:
        command = ['git', 'clone', '-q', '--depth', '1']

        '''
        Install ComfyUI at the verified official release tag.
        安裝 ComfyUI 時指定官方穩定版標籤。
        '''
        if folder == COMFY:
            command += ['--branch', COMFY_REF]
        subprocess.run(command + [url, str(folder)], check=True)
    if folder.name == 'ComfyUI-MultiGPU':
        print('MultiGPU v2.6.4 已停止維護，與新版 ComfyUI 的相容性仍須在 GPU 上驗證。')

    elif folder != COMFY:
        '''
        Keep user modifications intact rather than force-resetting them.
        保留使用者修改，不進行強制重設。
        '''
        if not subprocess.check_output(['git', '-C', str(folder), 'status', '--porcelain'], text=True).strip():
            subprocess.run(['git', '-C', str(folder), 'pull', '--ff-only', '-q'], check=True)
        else:
            print(f'略過已有修改的外掛：{folder.name}')

'''
Retrieve the official ComfyUI release even for an existing checkout.
現有 ComfyUI 倉庫也要取得指定的正式發行標籤。
'''
subprocess.run(['git', '-C', str(COMFY), 'fetch', '--depth', '1', 'origin', 'tag', COMFY_REF], check=True)
subprocess.run(['git', '-C', str(COMFY), 'checkout', '--quiet', '--detach', COMFY_REF], check=True)
ZIMAGE_CODE_UPDATED = any(
    folder in old_commits and old_commits[folder] != subprocess.check_output(
        ['git', '-C', str(folder), 'rev-parse', 'HEAD'], text=True,
    ).strip()
    for _, folder in REPOS
)

'''
Report all active upstream revisions.
顯示每個原始碼倉庫的實際版本。
'''
for _, folder in REPOS:
    print(folder.name, subprocess.check_output(['git', '-C', str(folder), 'rev-parse', '--short', 'HEAD'], text=True).strip())

installed = ROOT / '.zimage_kaggle_v045_installed'
import torch

'''
Store the installed PyTorch version as a pip constraint.
將既有 PyTorch 版本設為 pip 限制。
'''
constraint = ROOT / 'zimage_torch_constraint.txt'
constraint.write_text(f'torch=={torch.__version__}\n', encoding='utf-8')
if not installed.exists():
    '''
    Install the official ComfyUI dependencies.
    安裝 ComfyUI 官方依賴套件。
    '''
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-c', str(constraint), '-r', str(COMFY / 'requirements.txt')], check=True)
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-c', str(constraint), '-r', str(COMFY / 'custom_nodes/ComfyUI-GGUF/requirements.txt')], check=True)
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-c', str(constraint), 'huggingface_hub', 'hf_xet', 'requests'], check=True)
    installed.touch()

'''
Confirm that installation is ready.
顯示安裝完成狀態。
'''
print('ComfyUI installed')
print('✓ 第 2 節完成：套件與圖片生成環境已準備完成。')


3. 下載模型


In [ ]:
'''
Import the official Hugging Face downloader.
匯入 Hugging Face 官方下載工具。
'''
from huggingface_hub import hf_hub_download
MODEL_FILES = [
    ('unsloth/Z-Image-Turbo-GGUF', 'z-image-turbo-Q6_K.gguf', 'unet/z-image-turbo-Q6_K.gguf'),
    ('Comfy-Org/z_image_turbo', 'split_files/text_encoders/qwen_3_4b_fp8_mixed.safetensors', 'text_encoders/qwen_3_4b_fp8_mixed.safetensors'),
    ('Comfy-Org/z_image_turbo', 'split_files/vae/ae.safetensors', 'vae/ae.safetensors'),
]
for repo, name, relative_path in MODEL_FILES:
    destination = COMFY / 'models' / relative_path
    destination.parent.mkdir(parents=True, exist_ok=True)
    if not destination.is_file():
        if destination.is_symlink():
            destination.unlink()
        cached_file = hf_hub_download(repo_id=repo, filename=name)
        destination.symlink_to(cached_file)
    print('Ready:', destination.name)

print('✓ 第 3 節完成：模型權重已下載並準備完成。')


4. 啟動模型服務


In [ ]:
'''
Import the tools for managing a persistent local server.
匯入管理常駐服務所需工具。
'''
import time
import requests
import torch
assert torch.cuda.is_available() and torch.cuda.device_count() >= GPU_COUNT, '可用 CUDA GPU 數量不足；請檢查 NVIDIA 驅動、PyTorch 與 GPU_COUNT。'
BASE = f'http://127.0.0.1:{COMFY_PORT}'
LOG_PATH = ROOT / 'zimage_server.log'

'''
Check whether the model server is already running.
檢查模型服務是否已經啟動。
'''
try:
    running = requests.get(BASE + '/system_stats', timeout=3).ok

except requests.RequestException:
    running = False

if running and globals().get('ZIMAGE_CODE_UPDATED', False):
    '''
    Require an explicit restart for processes not owned by this setup cell.
    服務可能不由此 Cell 管理，因此要求先關閉舊行程。
    '''
    raise RuntimeError(f'ComfyUI 程式碼已更新；請先停止 {COMFY_PORT} 連接埠的舊服務，再執行本 Cell。')

if not running:
    log_file = LOG_PATH.open('w')
    server = subprocess.Popen(
        [sys.executable, str(COMFY / 'main.py'), '--listen', '127.0.0.1',
         '--port', str(COMFY_PORT), '--default-device', str(MODEL_GPU_ID), '--highvram', '--cache-classic', '--preview-method', 'none'],
        cwd=str(COMFY), stdout=log_file, stderr=subprocess.STDOUT,
        start_new_session=True,
    )
    log_file.close()

'''
Wait for ComfyUI to complete startup and register custom nodes.
等待 ComfyUI 啟動完成並註冊自訂節點。
'''
for attempt in range(120):
    '''
    Check whether the server responds.
    檢查服務是否能正常回應。
    '''
    try:
        if requests.get(BASE + '/system_stats', timeout=3).ok:
            break

    except requests.RequestException:
        pass
    time.sleep(2)

else:
    raise RuntimeError(f'ComfyUI 啟動失敗，請檢查 {LOG_PATH}')

node_info = requests.get(BASE + '/object_info', timeout=60).json()
required_nodes = {'UnetLoaderGGUFMultiGPU', 'CLIPLoaderMultiGPU', 'VAELoaderMultiGPU'}

'''
Detect missing custom nodes before submitting a workflow.
在生成前先檢查是否缺少自訂節點。
'''
missing_nodes = required_nodes - node_info.keys()
assert not missing_nodes, f'找不到雙 GPU 節點：{sorted(missing_nodes)}；請檢查 {LOG_PATH}'
print(f'GPU {MODEL_GPU_ID}:', torch.cuda.get_device_name(MODEL_GPU_ID), '｜影像生成')
print(f'GPU {AUX_GPU_ID}:', torch.cuda.get_device_name(AUX_GPU_ID), '｜文字編碼與 VAE')
print('ComfyUI ready:', BASE)
print('✓ 第 4 節完成：雙 GPU ComfyUI 服務已就緒。')


5. 生成設定


In [ ]:
'''
Describe the desired image.
設定圖片內容。
'''
PROMPT = '''

'''.strip()
WIDTH, HEIGHT, SEED = 640, 960, 42

'''
Preserve the original Turbo sampling parameters.
保留原本的 Turbo 採樣參數。
'''
STEPS, CFG = 8, 1
OUTPUT_NAME = 'zimage.png'
print('✓ 第 5 節完成：圖片參數已設定（尚未生成）。')


6. 生成、預覽與下載圖片


In [ ]:
'''
Serialize notebook generation with optional remote FIFO GPU jobs.
讓 Notebook 手動生成與選用的遠端 FIFO 共用 GPU 鎖，避免同時推論。
不必先啟動 ngrok 或任何 HTTP API。
'''
import threading
GENERATION_LOCK = globals().setdefault('GENERATION_LOCK', threading.Lock())
with GENERATION_LOCK:
    import uuid
    from IPython.display import FileLink, display
    from PIL import Image
    assert WIDTH > 0 and HEIGHT > 0 and WIDTH % 64 == 0 and HEIGHT % 64 == 0, '寬高須為正整數且為 64 的倍數。'
    import secrets
    RUN_SEED = secrets.randbelow(2**32) if SEED == -1 else int(SEED)

    '''
    Require a supported seed before building the inference workflow.
    在建立推論工作流程前，先檢查種子範圍。
    '''
    assert 0 <= RUN_SEED < 2**32, 'SEED 必須介於 0 至 4294967295，或設為 -1。'
    print(f'本次生成 seed：{RUN_SEED}')
    workflow = {
        '1': {'class_type': 'UnetLoaderGGUFMultiGPU', 'inputs': {'unet_name': 'z-image-turbo-Q6_K.gguf', 'device': f'cuda:{MODEL_GPU_ID}'}},
        '2': {'class_type': 'CLIPLoaderMultiGPU', 'inputs': {'clip_name': 'qwen_3_4b_fp8_mixed.safetensors', 'type': 'lumina2', 'device': f'cuda:{AUX_GPU_ID}'}},
        '3': {'class_type': 'VAELoaderMultiGPU', 'inputs': {'vae_name': 'ae.safetensors', 'device': f'cuda:{AUX_GPU_ID}'}},
        '4': {'class_type': 'CLIPTextEncode', 'inputs': {'text': PROMPT, 'clip': ['2', 0]}},
        '5': {'class_type': 'ConditioningZeroOut', 'inputs': {'conditioning': ['4', 0]}},
        '6': {'class_type': 'EmptySD3LatentImage', 'inputs': {'width': WIDTH, 'height': HEIGHT, 'batch_size': 1}},
        '7': {'class_type': 'ModelSamplingAuraFlow', 'inputs': {'shift': 3, 'model': ['1', 0]}},
        '8': {'class_type': 'KSampler', 'inputs': {'seed': RUN_SEED, 'steps': STEPS, 'cfg': CFG, 'sampler_name': 'res_multistep', 'scheduler': 'simple', 'denoise': 1, 'model': ['7', 0], 'positive': ['4', 0], 'negative': ['5', 0], 'latent_image': ['6', 0]}},
        '9': {'class_type': 'VAEDecode', 'inputs': {'samples': ['8', 0], 'vae': ['3', 0]}},
        '10': {'class_type': 'SaveImage', 'inputs': {'filename_prefix': 'zimage', 'images': ['9', 0]}},
    }
    response = requests.post(BASE + '/prompt', json={'prompt': workflow, 'client_id': str(uuid.uuid4())}, timeout=120)

    '''
    Surface validation errors instead of silently waiting.
    若工作流程驗證失敗，立即顯示錯誤。
    '''
    if not response.ok:
        raise RuntimeError(response.text)
    job_id = response.json()['prompt_id']
    start = time.monotonic()
    attempt = 0
    while True:
        record = requests.get(BASE + '/history/' + job_id, timeout=30).json().get(job_id)
        if record:
            if record.get('status', {}).get('status_str') == 'error':
                raise RuntimeError(str(record['status']))
            images = record.get('outputs', {}).get('10', {}).get('images', [])
            if images:
                break

        '''
        Print periodic status in the cell output without assuming a step percentage.
        定期顯示已等待時間，不捏造採樣百分比。
        '''
        if attempt % 5 == 0:
            print(f'生成中… {time.monotonic() - start:.0f} 秒', flush=True)
        time.sleep(2)
        attempt += 1
    image_response = requests.get(BASE + '/view', params=images[0], timeout=120)

    '''
    Reject failed downloads to prevent corrupted files.
    防止下載失敗卻仍儲存不完整圖片。
    '''
    image_response.raise_for_status()
    filename = Path(OUTPUT_NAME).name
    if not filename.lower().endswith('.png'):
        filename += '.png'
    OUTPUT_PATH = ROOT / filename
    index = 2

    '''
    Preserve earlier generations by choosing an unused filename.
    選擇未使用的檔名，避免覆蓋先前作品。
    '''
    while OUTPUT_PATH.exists():
        '''
        Add a numeric suffix for each additional image.
        在重複檔名後加入編號。
        '''
        OUTPUT_PATH = ROOT / f'{Path(filename).stem}_{index:03d}.png'
        index += 1
    OUTPUT_PATH.write_bytes(image_response.content)
    print(f'完成：{OUTPUT_PATH.name}｜{time.monotonic() - start:.1f} 秒')
    display(Image.open(OUTPUT_PATH))
    display(FileLink(str(OUTPUT_PATH)))

assert OUTPUT_PATH.is_file() and OUTPUT_PATH.stat().st_size > 0, '找不到有效的 PNG。'
print('✓ 第 6 節完成：圖片已生成並顯示預覽與下載連結。')


7. （選用）啟動 ngrok 與 FIFO 遠端 API


In [ ]:
'''
Require the first configuration cell before enabling the optional remote API.
選用遠端 API 啟動前，必須先執行第一格共用參數設定。
'''
assert all(name in globals() for name in ('REMOTE_PORT', 'REMOTE_SERVICE_ID', 'REMOTE_ROOT')), '請先執行第 1 節共用參數設定。'
import zlib
import math
import re

import queue
import copy
import secrets
import io
import json
import os
import shutil
import subprocess
import tarfile
import tempfile
import threading
import time
import urllib.request
import uuid
import zipfile
from email import policy
from email.parser import BytesParser
from getpass import getpass
from http.server import BaseHTTPRequestHandler, ThreadingHTTPServer
from pathlib import Path
from urllib.parse import urlsplit, parse_qs

REMOTE_KIND = 'image_text'
REMOTE_FIELDS = ['task', 'prompt', 'width', 'height', 'steps', 'seed', 'cfg']
REMOTE_IS_LTX = False
REMOTE_SCRIPT = zlib.decompress(b'x\xda\x8dYkS\x13Y\x1a\xfe\xce\xaf8_\xb6Bf4\xc3E\x9d\xd1\x1a\xb6\xcaUDj\xbc\xb0\x02\xbb[kY\xa9NrBz\xe8tg\xfa\x022\x14UqJ\x04\x84\x08\xbb\n\xde\x18\xf1\xca\xa0\x8eft\xb4DD\xfc/\xbb\xe9N\xf2i\xfe\xc2>\xef9\xddM\'\xc6\x99\x9d\x0f#\xdd}\xcey\xef\xcf\xfb\xbc\'\xb1X\xac\xad?_0L\x9b\xd9\x86\xa1Y,k\x98\xcc6\x95\xf4\xa8\xaa\x8f0E\xcf\xb0\x8cj\x154e\x82\x1e\xed\x1cg#\\\xe7\xa6b\xf3\x0cS\xf3\xca\x08O\xb4\xb9\x0bew\xfaQ\xed\xc3\xfb\xda\xdb\x87\xf5\xe2Kwk\xbd6;S\xbfW\xae>\xdcrWW\xaas3\xd5[\x17\xdd7\x8f\xdc\xe97\xff-\xfe\xd0\x16\x83<U\xcas\x1c5\xd3\x965\x8d<\xeb\x1f\x98\xb0s\x86\x9e\xf0E1\x7f\xc11U\xe3\'T}tO\xa0\x83\\=\xd0\x7f"X\xd1O*\xb4\x89C\x8f\xe4xzT\xea\x84\xe5y\xae[\xaa\xa1[,\xc5a\x10g\x96\x93\xca\xab\xb6-lb\xdf\x1a\xa9D\x9b\xf7\xe4\xbe\xb7\xf6Hj\xe8\xfe\xb2\xe5\x967k\x1b\xcf\xdc\xe7\xb7\xbc\x1bewi\xdd[\x9d\xf3\x96g\x03\x85\x15\xcb\xe2\x90\xf6\xf7\xfe\xa3C\xc7\xd9\x9fY\x87\xf0\xcb\xf1\xde\xfe\xbe\xe3C\xe1\xa3\xfc\xf8\'v`\x1f\xeb\xe9i\\\x12\xbe\xdc\xc3bn\xf9\xe7\xfa\xd3\x1b\xf5\xbb\xb3\xd5\x1f\xb6\xbcg\x0f\xbc\xe5W\xde\xf2fe\xf3\x1a\x1ei\x15\xf9\xaaX\xc2+\x88\x8eI\xc3\xfa|\x87C\xf1\xac\xc9\xad\x1c\xeb\xee\xda\x9bRmfq\xc4\xc0\xd0\xb5\t6\x9e\xe3:\xdb\xdb\xc9T\x8b\x99\xfc;\x87[\x88\x0e\xe2\xb2\xf8\x04VH\xa3\xe8\xabw\xf3\x07o\xae\xe4\xbe\xdb\xaa>\x9d\xf7V~\x81\xa4\xe0\xa4\xfa\xad\r\xef\xf1Zu\xe3\xb9\xfbl\xa9)F\x16O\x9b\xdc\xb6\xa4&g\xb8ehc\x1cB\xb9\x94^\xe0&3\x1d\x9d\x8d\xab\x08\x9fc3c\x8c\x9b\xe3\xa6j\x07\xb9\xa2\x1b6O\x19\xc6(V\x0b\xdf\xc3\xeb\xe5E\xef\xe7{\xd5kk\xde\xec\x92\xf7b\x0b\xaaU6\x8bnq^\n\xffm{\xa1\xf2\xfeRe\xb3T[\xbf\xe4\x96\x9f\xb2S\xc1~r\xcb\x955\xf7\xa7yiN\xa0\xe4\x99\xe1S\xc9\xc1\xde\xde\xa3\xac\'P4a\xc2\xef)\xae\x19\xe3\xed]\x9f}\xd6\xdd\x15gj\x96\xc9%=\xe4\x04\xaeY\x9c\xa9\xba\xddN\xef\xe2\x81Y\xdf9\xaaI\xfe\xb5\x9c\x02Y\r\xcb\x84y~\xee\xa4\x1cU\xcb\x04&\xa9z\x96\x9b\\Os6n\x98\xa3Y\x08\x82\xa7W7|\xb7^\xd9\xa8=]A\xb2W\xde\xafz\xaf/T7\xe6\xdd\xb9\x12\x8cr\xa7ge\xbaI3\xab\xe5\x8b\xeej\xa9)\xb9:\xd8\xd7=,4\xe8k&\xd4G\xc2\x88G\xf7\xc34R\xa6\xf2\x0e\x81\xdb\xc6\xca\xda\xcc\xafl_\xd7\xc1}\x07\x0f|\xd9up?\x04x\xb3+\xf0\x0c\xe5\xd0\xde\xce\xdd\xcc\x19\xcc\x19\xe3Bi%m;\x8a&\x8d\xb2\x0c\x18J^B%Q\xec\xd2\x8a\x0eC\x918\x05\xd3\xc88i\xac\xd0\x90lf\xa2\xcd/\xe1\xf2z\xfd\xd6\xed\xca\xfb\x0f\xd5k\x1b\x88B\x18\'o\xe5me\xe7C\xe5\xed\xbc\xbb\xb3P\x9f)U\x17w\xfc,\x12\xa1\xad\xbe^\xf2\xee\xac\x06\x16\x16L\xf2x6\xe6\xad\xfe\x1c\x06_(\xf3\xdb\xf6\xad\xc9\xc0\xe4\xa9\x98\x1f\x8e\xa3<\xab"\xc1HoK\xc9s\xd6\xd77|,t\xb6H5\xa6\xda\x16\xd3\x0c%\xc3M\xe0\x95z\x1e:\xdb\x06\x0e,(\xa2N\xfa\x06\x86\xadD[\xe5\xc3\x8f\xd5\xe5\x9b\xc8\x1aH\x95gD\xe3B\x99\xb6\xb9\xee\xfe2\x83ds\x97\x16\xbc\x8d{\xee\x9dy\xf76%\xa4;{\xa9>]\x82\x8b\xdd\xe9\xc7\xee\xf6k:.0$\xd4\xa2\x87M\xb61\xfc\x17\xeb\x8c\x1db\x93\xb1\xb4\x86 &\xed\x89\x02\xc7clX\xe7\xf6\t\xa1\x1d\x89=\xe9h\xb6\x8a3b\x88\xa5\xaa\x17\x1c\xdb\x12[\x1c,J\xea\xb0\x8fv|\xbfW\xe0\xd6^\xdb1S\xc6\xde\xbf\x1eH~\x93\x18\x19q\xb2\xb4%\xc3\xc7\xd44-\xca\xc6\xd2NF94y\xf2\xf4\xd1\xde\x13I\x1c\x98\xec\x87\xcf\xa6\xa6\xf6HE\xbaZ(r\xe4D\xff\x80T\xe4\x13J\xa45\xb5\x10*\xf1\xdd8\xd7\x93\xdd\xc9}\xa9d\xb6\xf0U2OnMXJ\x96\xdb\x80R\xc3\xb4hkp\xb0\xe6\xe4U]\xe9j\xa9\xdf\xe1\xe1\x7f|\xac]w\x0b\xed\xfev\xb8\xf7w\x95\x1bSx\xa8\x9b\xc2\x9bU\xf9\xff\xe4\xee\xfb\x84W\x86\xf8y\xbbWO\x1b\x19\xde$\xd4\xc6\x07\xfc1p\xe6\xf4\xc9\x81!|"\x0f\xe1\xf9l\x8c\x8c\xed8\x17\x1e\xbc\xbf\xd5\xc1\x86\x9e\x01\xf6\x19:\xb0\xe2\x9f\xdc4N;v\xb3\xbf#+\xc4\xa9\xfb\x1aO=\xd0\xe2\xd4\xde|\xc1\x9e\x18<\xda}\x02\xa9\xad\xdb\xa2\xe15\x9d:\xaef\xec\x1c\xfe\x12\x1d\x08\x9fr\\\x1d\xc9\x91\x15\xb2\xfd\xe0MJ\xb1\xd3\xb9\xa4\xa5~O\'v\x86\xe2\xbel!\xee$|\xa2\r*\xf9\x82\x06\x1d\x0f;\xa6r\x0c\xf9\xde$\xd0\xca\xa9Y:\xbf\x1b\xaf\xf3\xb4^\xd8\xd2\xd9h\xcbW-\x0e\xffF\x1c\xcc\xcd\xe6\xf3\x00\x05\xf8#@\x02|D\x07+\xd0\xb7\xc1\xa1\xde\x81A\nC\x96\xfcu\xe4X\x1f}\x93g\x84\xb9\x81\x9e\x98\xccS\n\xd1&:\xd9J\xe7x\xc6!1\xf8j\xa9\xb4Zf\x8cn\xa8\x96\xf0@\x83\xde_\n\xbd\xf1\xaa`X\x08\xce\x18\x8fD\x06ou>\xa2\x84o\xf7\x07o5\x11\x8d\xa4([\xf1\xe5@\xa3\xf5\x07[\'\xfcQ\xde"\xe7\xa4A\x968\xe6\xab@\x00\xb2_\xbc\xe8n<\xb7\xb3\xa3\xc5\xc1\x83\xca\x18o\x95\x18Y\xf0\'\xf2R\xb2`\x02S\xcf\x0b\xa4Q\xc3u\xf4\x87\x14z0\x941\xe57\x0e\xc1\x96d\xbf\x13\x84\n\x8c\x89 \x96^\xf0\xf3p4u\xc3#F>;1\xdc\x0f\xdc5\xc7\xa8_\x00L%\xb4K\x9c\xad\x17/\xb8;E\xf7\xcdKwy\xcd\x9d_&\xba\xe1\xef\x10\xd8[B\xd3\x00\x0b\xa9m\xcf\x81<J\xf8\xf5VK\xee\xfc\xcd\x00k\x11\xd6\x02\x08\x1c\x07\xd6\xfa\xa4\xc6J Bv\xfb_\x0e\x0f\xf6\xb2\xcfY\xec\x0b4,\xd4\x06t\xff\xd62\xf4\x9e\xc9\x98\xff|(l\x16\xb2~E\x9c(\xbf,\xdbl\'\xce\x99\xa0\xff\xedk\x8f\xc7\xa7\xf60\x1bL\x11\xcc\xa5\xa7\xb3\xab#\x1e\xd8nf\x15\xb4\xf71ES3\n\x15+\xe3\xa6\t\xd4A\xebG\x86)\xe0\\Yf\x91gm\xa2^\x8a*\x89Mm\xfeQ\xb4\xbf\xd4\x1f_\xaf=\x9bs\x1f\xbe\xf0\x96\xaf\xc3^p\x03\xb7\xf4\xab\xec\xa8\xf5\x85r\xed\xc9\xc3\x90ge\x89%\xb1\xc0\xda\x841z\xa8M\x86\x1a\x1f\xe9\xdf\xb0\x89\x87\x0eI\x19\x99\t\x8aF\x8ek\x05P]eD7,\xe2Z\x19033\xa0Z\xbe\xd2\tq\x86\x14\\\xd9\xde"\x17\xdf\xdcpo\xdf\x01\xc1"^R\xba\xe6^~L$\xec}\xa9Z.\xd6\xdf\xdd\x91D\x82\xba\xe0\xf2b\xfd\xde\x02i\x19U\xc6TPA\xec\x8c\xa3\x93\xdfzIB{\xa89Ag\xe0C$\xa4l\xe0\x92v\xa3AS\x06\xa9(A[\xcd\xaa\x94-\xb5\x8d\x1b\xf5\x85\x8bQF \xddG\x1c\xe3\xcdF\xed\xe6Z\xe0 lD\xf8D\x16\xf8\x82(\xdc\xed\xf1\xb3~\xbc)\xb6\xe7\xa4\xd8!\x1a[d\x92jJ\xc1\x82T\x7fX\xa1 \x92\xc6\xad\xc4zs\xc5\xda\xe5\x17 \xc7\xf5\x95\xab\x81L\xcbV\xc0\xc8z\xe4\x9e\xbc\x81\x00\x01\xb2\xd3\xed\xf1\x08\xad\xc2\x12\xdb\xb1@\xbd\x91\tA\x16\xc8\xd9\t\xe2\x07N\xf51d\x15\n1\xd1VE\x1a\xecL\x8bW\xb5\xedMwf\x0b\xa2h\xccX]\xf3\xd3A\x8eL\x1b\x97\xbd\x0b\xe5\x90\x12\xc2e0\r\x1at\xb4I\tC\xa6\xc3\x9b\x12\xe3\x0c\xe5"IK;&\x18\xa9\xbd\xcb\x91r\xa8P\xc3\x9c\x90\xb1\xaf=/\xba\x8b+\xd5\xdb\xcfAE\xa3\x19J\x94z\xedM\xed\xdeB\xf5U\x11.\xf9(\xd2@*3\x13-\xbe\x11\x1e\xa9=_\xc4\x171<\xc9\x08\xed\xd6RwG\xdc\x8f\x91\xd8#?\xc7\x1b\x95\xef\xd7\xad\x02O\xdb,m\x10\xf6Q\x82\xc0uY\x05\xa6\x8aT\xf1\xf3\xd6\x9f\xd1\x80!\xcf\x17(T\xb3+\xb2\xa2\xe4D\tS\x9a\xb5F5I\xc5}_E\xbfI\x9f\x89\xa9\x86\x9f\xe7i\'Z\xdab\xae\xf4\xe1)\x11\xae\x96\x01\n\x81NRa\xe9\xb4\xdd"n%&TC8 &s\x05@59\x15\x8f\xbeI\x02\x91bq\x9aLbB\x8fXD\xeb\xe6#\x05\x14\xd8FA\x0e{\xa2\xb6\x04\xee\x12q\x87I\x16\xe6MiL\xa2a\x8b\xc4T\x14\xbc{\xf7\x85\xd4\x19\xe9\'\xd1\xc8\xbd\xb0\xea=\xbb\x1f5\xa1\x95\xcc\x16\x15O8*\xcd;\x1b\x98v.\x1e\xff\x94\xc3\x15\xff\xa2\x80\xe5\xb9\xad\x00P\x15iA\xa42A\xf7U+\xc7\xad]\xcdw\x87\x88\x1f\xd70_ \x7f\xdd\x9d\xebrT\xaf\xfd:\x83Z\xf9\xa4\xe3ES\x13Y\xbb\xeb~Y\x89\r\xfeG\x1b\x8d<\xf9\xadp\x0f;{\xee\x13f\x08\xcf\x17\x0cM\x13\x17\x08\x16\x86(\xe8\x8d\x7fw;$foe\x0c\xf9\xab\xa44\xbek\x88\x7f\xbd\x80\x04^,K+\xc8\xfd\xc2.\x89\x0b\xbf\x93AR\xab?H\tpN \x8f\xc3\xa9\x1fd\x8cq\x9d\xa6\xa2pT\x95\xf73\r\xe9p\xe5Q\xed_\xd7+\x9b\xf3\xe8\xbb>\xea\n\r\xff(\x0bR&WF\x1b+x\x80\x86:\xba\x01P\x8d\x8c\x9a\x0e\xf0P\x95\xb9\x99\xe6\x9a\xe6#`x9\x00\xc6Bc\x03]\xc0\x10S\xa3\xadi\xc0V\xa8c\x14\x14\xe11\xe9\x1f\x89\xcb\x921x\xa5\xc5z\xf1\xaew\xe5\x9e\xb7\xf1\xa0zs\x07s\x9aW\xbe\xd6\x02\x01\x02\xf4\xfc\x13\xdb/n]>\x81\x05G\xfd\xcb\xa6\xa0WX\x9c\xe8y\x00\xe9\x91\nCh\x83l\xa5N\xbf\x1b\xdb\ri\'\xdc\x89iX*.q\xb6V\xbc^{\xf9\xae\xfa\xd3\xbf\xe5ENK\xf1\xc1P,\xc3P\xd9|\xf6\x9f\xe2:\x9bl\xee8l/\x13\xcd\xe8P\xa2#;\xc5p"\xb24\xab9V\xae\x87:B\x13\xa8\x0e(\x0e\xd1\x03yk!\xee_\xd0\x92\x83n\x10\x00\xb94@z\x97rs\xf6\x1d\x11\x81K%4D`m\xed\xf1}\xbf\x0f\xcf\x17\xbd\xe9\xf9f\xef\n\xf5,\x8d\xf3B{W\\\xbc\t\x9c\xfdy\x0f\xeb\xf4\xe7w?\re"\x84\xf8N\r\xb0\x11f\xdd\x9d;\xbb\xd8*\x8a\xc3OI\x01\xf6D\x18\xb1g\xf7>\n\x89\x92lE\n\x1b\xfa\xd2\x98\xcaiZ\xa1\x9b\x80\xbc\xd5#K\xe8,\x11\xea\x8f\x89\xde\x19\xfe-u \xbf\xe9\x04\xc5cQ%\x81/\x8fQG\x05\x86\x98N\x81\xb4\'*\r\x80\xaa\xdfx\t\xcc\x94\xf5#[\x91[zWyWr/\xbet\x9f\xdd\xa0\xbb\x04\xe8\xbe\xfcj\xb7\xa8>V=!\x105\x89\x18%e\xcd\x84\xb4B\xd1\xc1"\xbe\xe7>\xd9\xf3\xef\xf1X@\xe2\xc5\x85"\xd7\xb1/\xddH0\xbc\xcd\xe9\xea\xd2%\x9f[<\xb9\xe6.\x95*\x9b\xeb\xd5\xd7/\xe8bM\xa8E\xd7B\x11O\x86\x07\xf6 a\xec\\\xfb\xe9\xe1\xa1\x81\xe1\xa1\xe4\xa9\xc3\'{\xe3\t\xfa \xd5\xe9\xd5-\x07i\xe4\x17q\xb0\t(\xa7\x8d+\x13\x16\xcb\x05\x14\x8fTA\x9a\xc9\xfbV\xd0\x9d\xfb[\x95\x0f?6h#\xee\x8e\xc4:w\xae,\xdf5\xb1\xdf\xe0\xf4\x04\xd8\x0b7\xc1\x1b8\n\x91p\xa3=\x96(`T\x8e7Q\x9f\xc3\x85\x02V\xc8\x1a\xb5\xe1\x17\xc5\xcc4\xeb!\xa8\xcf\xcc\x13L\x1e\xb5\x07\xab\x95\xcd\xcb\x1f\xcb\x8f\x1e\x18\xfa\x04i,EF\x88,"\x87\xf1R`\xdb7\xca\xc8\x08\xa6IKp-\xc2\xb2\x8c\x8aN#\xc9\x16\xcd?\xe2\xdaKz\x9d\xe0A.g!]\x01:\xf8T\x8b\x0e\xf7\x1d?px\xe88bq\xe6\xf4\xe9!\xf6E\xa8\x89/_\x10Q\xcb\xc9b|c\xba\x93O\x01n}\x9e\x99q0\xa2\xa7\xe9\xbe+\x8c\x8d,\xa9\x8e\x8e.V_\x99w\x7f\x9a\x97D\xda]Z\x80\xc5M)\xa9g\xf8y\x08\xed\x92b\x06`"a\x1d\xe3\x8a\x89\x81\xc9\x8ctg\x8b\xa5&X:g`6\x16\xbf\x070Gw\xacHZ\xa2&.lzWg\xbc\xd5\'\xe1%\xa1\xf41`\xa5~\xe1\x83;M\x97\xb9\xb5\xab\xf3\xee\xf4,\x90\x11Np\xaf^\x08\xef\xd3\x04\xd4F\xfc\x90\x10\xd3%J\xa29\xe0\x99\x0c\xda\x06\x1c\x00\xfb\xd3\x81?\xc8\x0b\\I\xe7\x98\x92\x91\xb7*\x8a\x16\xedy\x08\x00f\xcc\xda\xc3\x19\xa9\x0e\xf1\x88\xcbw1l\xee\x8e\x17Q\t\xad\x83\x11\x9b\x14%\x12\x18\x1bO\xa0$\xf3S\xc9I\xe1\xbfC\x1d\xdd\x99)?Y\xa2G\x9d4\xc6x0-\x0b\x04\x96\x81\xa3\xd6d\x03\x8c\x99H4\xe2\x0b\x1a\xba*F9\xf2\xa7\x9f\xb0\x18"\x97J\xd5\x8d\x17\x80\x14\xa9;\xcd\x8eo\xb7\xab[\x0b\xee\x8d\xeb\xb5\xc7\xeb@\x1e\xff\xbe\xbc\xa5\x112\xac\xbbP\x1c\xcea:\x1f\x17\x05\x00\xad\xfc\x9c\xa4\x9e \x7f\xbe\t\xc6v\x1fpAs\x9f\xc2K\xc1:\x1fB\x1eoE\xc6\xa3h\xc0\xe8\xba\x9f\'S\x136\xb7\xda\x9b\x80\x0e\xed\x94.I\xa2S\x93\xdf\x0e\x82yL`\x9a\xbc\xe8\x1dk\xcc)\xd9L\x85^h\xa6`\x01\xb5\xd9\x19\xa9J#\x80\x04]T\xf6\x0c\xbaO\x8e*G\x87M\xfd\xf6~\xf5w\xdaj\xa7\xdfV\xe3a%P\x0b\x11J\xf9\xbc4\xe0\xaf>\xb5\t~\xd3\x00\xd4\xdd~\x05>Eu\x1e\xfeN\x816^\xbf\xbbT[\x7f\xdfL\xae\xe8l\xffG\xacvqU\x930\x00a\xed\x11e\xe3\xa1\x06\xc6\x18\x06e\xa4\xbbf\xa4\x91\xd1\xa0\x9c\xa3A6I7!H@\xfd\xc5\xa5\xca\xcem\x110x\xc4\xbb7+\x9bR\xbd\xf8\x00 \xd4,1\xf8!MP\xf7\x06\x99\xf1\xff\x01\x92\x9b\x16\xbd').decode('utf-8')
REMOTE_ALLOWED_TASKS = ['text']
REMOTE_MAX_IMAGES = 0

REMOTE_ROOT.mkdir(parents=True, exist_ok=True)

'''
Share existing state when the API cell is executed again.
重跑 API Cell 時保留原有工作及鎖定狀態。
'''
REMOTE_JOBS = globals().setdefault('REMOTE_JOBS', {})
REMOTE_JOB_LOCK = globals().setdefault('REMOTE_JOB_LOCK', threading.Lock())
REMOTE_ENGINE_LOCK = globals().setdefault('GENERATION_LOCK', threading.Lock())
REMOTE_QUEUE = globals().setdefault('REMOTE_QUEUE', queue.Queue(maxsize=REMOTE_QUEUE_CAPACITY))
assert REMOTE_QUEUE.maxsize == REMOTE_QUEUE_CAPACITY, 'FIFO 容量已變更，請先重新啟動 Kernel 及遠端服務。'
REMOTE_UPLOAD_SLOTS = globals().setdefault('REMOTE_UPLOAD_SLOTS', threading.BoundedSemaphore(REMOTE_UPLOAD_CONCURRENCY))
REMOTE_RESULT_TTL = REMOTE_RESULT_TTL_HOURS * 3600
REMOTE_RESULT_BYTES = REMOTE_RESULT_LIMIT_GIB * 1024**3
REMOTE_MIN_FREE_BYTES = REMOTE_MIN_FREE_GIB * 1024**3
REMOTE_STALL_SECONDS = REMOTE_STALL_HOURS * 3600  # 無限大代表不因等待時間長而拒絕工作

'''
Verify the CUDA backend and existing model without loading another model.
確認 CUDA 與已啟動的模型後端，不另外載入第二份模型。
'''
def remote_gpu_status():
    try:
        import torch
        if not torch.cuda.is_available():
            return False, 'CUDA GPU 不可用；請檢查 NVIDIA 驅動與 PyTorch，並重新執行模型初始化 Cell。', []
        qwen = REMOTE_KIND == 'image_multi'
        yue = REMOTE_SERVICE_ID.startswith('06_YuE2_')
        if yue:
            selected_gpus = [globals().get('GPU_ID', 0)]
        elif qwen and globals().get('QWEN21_STATE', {}).get('mode') != 'bf16_dual':
            selected_gpus = [globals().get('GPU_ID', 0)]
        else:
            selected_gpus = sorted({
                globals().get('MODEL_GPU_ID', 0),
                globals().get('TEXT_GPU_ID', 1) if REMOTE_SERVICE_ID.startswith(('06_MiniMax_', '06_HyperFlow_'))
                else globals().get('AUX_GPU_ID', 1) if REMOTE_SERVICE_ID.startswith(('06_HeartMuLa_', '06_LTX-2.5_', '06_Z-Image-Turbo_'))
                else globals().get('TEXT_GPU_ID', 1) if qwen else 0
            }) if not qwen else sorted({globals().get('GPU_ID', 0), globals().get('TEXT_GPU_ID', 1)})
        available = torch.cuda.device_count()
        if any(not isinstance(index, int) or index < 0 or index >= available for index in selected_gpus):
            return False, f'需要 cuda:{selected_gpus}，但實際只有 {available} 張 CUDA GPU。', []
        devices = [f'cuda:{index} {torch.cuda.get_device_name(index)}' for index in selected_gpus]
        if qwen:
            state = globals().get('QWEN21_STATE', {})
            if state.get('pipeline') is None or state.get('manager') is None:
                return False, 'Qwen Pipeline／GPU 記憶體管理器尚未初始化，請先執行模型初始化 Cell。', devices
            if state.get('device') != f"cuda:{globals().get('GPU_ID', 0)}":
                return False, 'Qwen Transformer 不在設定的 GPU 上。', devices
        elif REMOTE_SERVICE_ID.startswith('06_HeartMuLa_'):
            pipeline = globals().get('pipe')
            if pipeline is None or pipeline.lazy_load or pipeline._mula is None or pipeline._codec is None:
                return False, 'HeartMuLa 雙 GPU Pipeline 尚未完整載入，請先執行模型載入 Cell。', devices
            for name, module, index in (('HeartMuLa', pipeline.mula, globals().get('MODEL_GPU_ID', 0)), ('HeartCodec', pipeline.codec, globals().get('AUX_GPU_ID', 1))):
                parameter = next(module.parameters(), None)
                if parameter is None or parameter.device != torch.device(f'cuda:{index}'):
                    return False, f'{name} 未位於指定的 cuda:{index}，請重新載入模型。', devices
        elif yue:
            gpu_id = globals().get('GPU_ID', 0)
            process = globals().get('SERVER_PROCS', {}).get(gpu_id)
            if process is None or process.poll() is not None:
                return False, f'YuE2 CUDA{gpu_id} 常駐服務尚未啟動。', devices
            if f'CUDA{gpu_id}' not in [str(value) for value in process.args]:
                return False, 'YuE2 常駐服務沒有使用預期的 CUDA 啟動參數。', devices
        else:
            base = globals().get('BASE')
            if not base and REMOTE_IS_LTX:
                base = f"http://127.0.0.1:{globals().get('COMFY_PORT', 8188)}"
            if not isinstance(base, str) or not base:
                return False, 'ComfyUI 尚未啟動，請先執行模型服務 Cell。', devices
            with urllib.request.urlopen(base.rstrip('/') + '/system_stats', timeout=4) as response:
                stats = json.load(response)
            comfy_devices = stats.get('devices', [])
            if not any(str(item.get('type', '')).lower().startswith('cuda') or
                       str(item.get('name', '')).lower().startswith('cuda:')
                       for item in comfy_devices if isinstance(item, dict)):
                return False, 'ComfyUI 沒有回報 CUDA 裝置，已拒絕 CPU-only 推論。', devices
        return True, 'CUDA 後端已就緒（不代表所有權重常駐 GPU 記憶體）。', devices
    except Exception as error:
        return False, f'GPU 檢查失敗：{type(error).__name__}: {error}', []

'''
Verify every global required by the selected Notebook generation engine.
驗證所選模型的生成程式所需變數與設定已完成初始化。
'''
def remote_inference_ready(task):
    namespace = globals()
    common = {
        '06_HeartMuLa_': ('ROOT', 'pipe'),
        '06_YuE2_': ('ROOT', 'GPU_ID', 'PORTS', 'SERVER_PROCS'),
        '06_Z-Image-Turbo_': ('ROOT', 'BASE', 'LOG_PATH'),
        '06_Qwen_Image_': ('MODEL_MODE', 'GPU0_MODEL_GIB', 'GPU_ID', 'QWEN21_STATE'),
        '06_LTX-2.5_': ('ROOT', 'ENGINE_PATH', 'COMFY_PORT', 'COMFY_LOG', 'COMFY_PROC',
                          'ltx_available_ram_gib', 'LTX_MIN_AVAILABLE_RAM_GIB', 'importlib'),
        '06_MiniMax_H3_': ('ROOT', 'BASE', 'COMFY', 'LOG', 'PORT', 'TASK_CONFIGS',
                              'H3_DESIRED_CONFIG', 'H3_ACTIVE_CONFIG', 'H3_MODEL_CHOICE',
                              'H3_ACTIVE_MODEL', 'DIT_NAME', 'TEXT_MODE', 'TEXT_GPU',
                              'QUANT_SIZES_GIB', 'ready', 'choices', 'check_choice'),
        '06_HyperFlow_H3_': ('ROOT', 'BASE', 'COMFY', 'LOG', 'PORT', 'TASK_CONFIGS',
                                'H3_DESIRED_CONFIG', 'H3_ACTIVE_CONFIG', 'H3_MODEL_CHOICE',
                                'H3_ACTIVE_MODEL', 'DIT_NAME', 'TEXT_MODE', 'TEXT_GPU',
                                'QUANT_SIZES_GB', 'ready', 'choices', 'ENCODERS',
                                'ENCODER_FILE', 'MODEL_FAMILY', 'WEIGHTS_IN',
                                'TEXT_LOADER_NODE', 'PROJECTION_FILE'),
    }
    required = next((names for prefix, names in common.items()
                     if REMOTE_SERVICE_ID.startswith(prefix)), ())
    missing = [name for name in required if name not in namespace or namespace[name] is None]
    if missing:
        return False, 'Notebook 尚未完成初始化，請先執行對應的模型／服務 Cells；缺少：' + ', '.join(missing)
    if task not in REMOTE_ALLOWED_TASKS:
        return False, '此模型不支援生成模式：' + task
    if REMOTE_SERVICE_ID.startswith(('06_MiniMax_H3_', '06_HyperFlow_H3_')):
        config = namespace['TASK_CONFIGS'].get(task)
        if not isinstance(config, dict):
            return False, f'{task.upper()} 尚未設定；請先執行 Notebook 對應的模式設定 Cell。'
        if namespace['H3_ACTIVE_CONFIG'] != namespace['H3_DESIRED_CONFIG']:
            return False, 'GPU／Offload 設定與已啟動 ComfyUI 不一致，請重新啟動模型服務 Cell。'
        quant = config.get('quant')
        if REMOTE_SERVICE_ID.startswith('06_MiniMax_H3_'):
            variant = 'ref2va_pruned' if task == 'r2v' else 'fl2va_pruned'
            expected = (variant, quant)
            expected_file = f'minimax_h3_{variant}-{quant}.gguf'
        else:
            variant = 'ref2va' if task == 'r2v' else 'fl2va'
            encoder = config.get('encoder_variant')
            expected = (variant, quant, encoder)
            expected_file = f'minimax_h3_{variant}_hyperflow_flash_14b-{quant}.gguf'
            if encoder not in namespace['ENCODERS']:
                return False, f'{task.upper()} 的 Encoder 設定無效，請重新執行模式設定 Cell。'
            if Path(namespace['ENCODER_FILE']).name != namespace['ENCODERS'][encoder][1]:
                return False, 'GPU 1 Encoder 與此模式不一致，請重新執行模型下載／啟動 Cells。'
        if namespace['H3_MODEL_CHOICE'] != expected or namespace['H3_ACTIVE_MODEL'] != expected:
            return False, f'{task.upper()} 尚未載入相應模型權重；請設定模式並重新執行模型下載／啟動 Cells。'
        if namespace['DIT_NAME'] != expected_file:
            return False, 'ComfyUI 使用的模型檔名與任務不一致，請重新載入正確模型。'
    elif REMOTE_SERVICE_ID.startswith('06_Qwen_Image_'):
        state = namespace['QWEN21_STATE']
        if state.get('mode') != namespace['MODEL_MODE']:
            return False, 'Qwen MODEL_MODE 與已初始化的模型模式不同，請重新執行模型初始化 Cell。'
        if state.get('gpu0_model_gib') != namespace['GPU0_MODEL_GIB']:
            return False, 'Qwen GPU0_MODEL_GIB 與目前載入設定不同，請重新初始化 GPU 記憶體管理器。'
    elif REMOTE_IS_LTX:
        if not Path(namespace['ENGINE_PATH']).is_file():
            return False, 'LTX 模型引擎檔案不存在；請先執行模型與 Workflow Cell。'
        if namespace['COMFY_PROC'].poll() is not None:
            return False, 'LTX ComfyUI 行程已停止，請重新執行啟動 ComfyUI Cell。'
    return True, '模型初始化與所選模式已就緒。'

'''
Validate client parameters before they occupy a FIFO slot.
在進入 FIFO 前檢查 client 參數，避免排隊後才因設定錯誤而失敗。
'''
def remote_validate_fields(task, fields):
    required = {'task'}
    if REMOTE_SERVICE_ID.startswith('06_HeartMuLa_'):
        required.update(('lyrics', 'tags', 'duration', 'seed'))
    elif REMOTE_SERVICE_ID.startswith('06_YuE2_'):
        required.update(('style', 'lyrics', 'duration', 'steps', 'lm_seed', 'seed', 'cot'))
    elif REMOTE_KIND == 'image_text':
        required.update(('prompt', 'width', 'height', 'steps', 'seed', 'cfg'))
    elif REMOTE_KIND == 'image_multi':
        required.update(('prompt', 'negative_prompt', 'sampler', 'reference_size',
                         'width', 'height', 'steps', 'seed', 'cfg'))
    elif REMOTE_IS_LTX:
        required.update(('prompt', 'width', 'height', 'seconds', 'seed', 'image_strength'))
    else:
        required.update(('prompt', 'width', 'height', 'seconds', 'steps', 'seed', 'ref_image_size'))
        if REMOTE_SERVICE_ID.startswith('06_MiniMax_H3_'):
            required.add('audio_shift')
    missing = required - fields.keys()
    if missing:
        raise ValueError('請求缺少必要參數：' + ', '.join(sorted(missing)))
    integer_fields = ('seed', 'lm_seed', 'width', 'height', 'steps', 'duration', 'reference_size')
    float_fields = ('seconds', 'cfg', 'image_strength', 'audio_shift')
    numbers = {}
    for key in integer_fields:
        if key in fields:
            raw = fields[key]
            if isinstance(raw, bool) or str(raw).strip() != str(int(raw)):
                raise ValueError(f'{key} 必須是整數')
            numbers[key] = int(raw)
    for key in float_fields:
        if key in fields:
            numbers[key] = float(fields[key])
            if not math.isfinite(numbers[key]):
                raise ValueError(f'{key} 必須是有限數值')
    for key in ('seed', 'lm_seed'):
        if key in numbers and not -1 <= numbers[key] < 2**32:
            raise ValueError(f'{key} 必須介於 -1 至 4294967295')
    for key in ('width', 'height'):
        if key in numbers:
            size = numbers[key]
            divisor = 64 if REMOTE_KIND == 'image_text' else 32
            if size < 1 or size > 4096 or size % divisor:
                raise ValueError(f'{key} 必須介於 1～4096 且為 {divisor} 的倍數')
    for key in ('seconds', 'duration'):
        if key in numbers and not 0 < numbers[key] <= 3600:
            raise ValueError(f'{key} 必須大於 0 且不超過 3600')
    if 'steps' in numbers and not 1 <= numbers['steps'] <= 200:
        raise ValueError('steps 必須介於 1～200')
    if REMOTE_SERVICE_ID.startswith('06_HyperFlow_H3_') and numbers.get('steps') not in (4, 6, 8):
        raise ValueError('HyperFlow steps 只支援 4、6、8')
    if 'image_strength' in numbers and not 0 <= numbers['image_strength'] <= 1:
        raise ValueError('image_strength 必須介於 0～1')
    if 'reference_size' in numbers and not 64 <= numbers['reference_size'] <= 2048:
        raise ValueError('reference_size 必須介於 64～2048')
    if 'audio_shift' in numbers and not 0.01 <= numbers['audio_shift'] <= 100:
        raise ValueError('audio_shift 必須介於 0.01～100')
    if fields.get('ref_image_size', 'match') not in ('match', 'max'):
        raise ValueError('ref_image_size 只支援 match 或 max')
    if REMOTE_KIND in ('video', 'image_text', 'image_multi') and not str(fields.get('prompt', '')).strip():
        raise ValueError('PROMPT 不可為空白')
    if task == 'r2v':
        matches = [int(number) for number in re.findall(r'<Picture\s+(\d+)>', str(fields['prompt']), re.I)]
        if any(number < 1 or number > int(fields.get('_image_count', 0)) for number in matches):
            raise ValueError('R2V Prompt 的 <Picture N> 編號超出已上傳的圖片數量')

'''
Report whether the model is ready for CUDA inference.
回報模型是否已具備 CUDA 推論條件。
'''
def remote_backend_ready():
    return remote_gpu_status()[0]

'''
A Python thread cannot safely terminate an already running CUDA inference.
Python thread 無法安全強制終止正在執行的 CUDA 推論。
'''
def remote_stalled():
    with REMOTE_JOB_LOCK:
        return any(record.get('status') == 'running' and
                   time.monotonic() - record.get('_start_clock', time.monotonic()) > REMOTE_STALL_SECONDS
                   for record in REMOTE_JOBS.values())

'''
Remove obsolete input files once the model no longer needs them.
模型不再需要輸入檔案時立即移除，避免參考圖片長期佔用空間。
'''
def remote_remove_inputs(job_id):
    folder = REMOTE_ROOT / job_id
    for file in folder.glob('*'):
        if file.name.startswith('input.') or (file.stem.isdecimal() and file.suffix.lower() in {'.png', '.jpg', '.jpeg', '.webp'}):
            file.unlink(missing_ok=True)

'''
Keep only finished results; never delete active jobs or in-progress downloads.
僅清理已結束工作，絕不刪除排隊、推論或正在下載的檔案。
'''
def remote_cleanup():
    now = time.time()
    with REMOTE_JOB_LOCK:
        finished = [(job_id, record) for job_id, record in REMOTE_JOBS.items()
                    if record.get('status') in ('done', 'error', 'expired') and
                    not record.get('_downloads')]
        finished.sort(key=lambda item: item[1].get('finished_at', now))
        sizes = {}
        for job_id, record in finished:
            folder = REMOTE_ROOT / job_id
            sizes[job_id] = sum(path.stat().st_size for path in folder.glob('**/*') if path.is_file()) if folder.exists() else 0
        total = sum(sizes.values())
        for job_id, record in finished:
            old = now - record.get('finished_at', now) > REMOTE_RESULT_TTL
            if old or total > REMOTE_RESULT_BYTES:
                shutil.rmtree(REMOTE_ROOT / job_id, ignore_errors=True)
                total -= sizes[job_id]
                record.update(status='expired', error='結果已過期或因儲存空間不足而清除，請重新提交。')
                record.pop('_path', None)
                record.pop('_last_frame', None)
            if record['status'] == 'expired' and now - record.get('finished_at', now) > 2 * REMOTE_RESULT_TTL:
                REMOTE_JOBS.pop(job_id, None)

'''
Sweep completed records periodically without interfering with inference.
定期整理已完成資料，不干擾單一 GPU worker 的推論。
'''
def remote_maintenance():
    while True:
        try:
            remote_cleanup()
        except Exception as error:
            print('API cleanup warning:', type(error).__name__, str(error)[:200])
        time.sleep(300)

'''
Serve only the generation API without exposing notebook controls.
只提供生成 API，不公開 Notebook 的管理功能。
'''
class RemoteHandler(BaseHTTPRequestHandler):
    def log_message(self, fmt, *args):
        return

    def send_json(self, code, payload, retry_after=None):
        data = json.dumps(payload, ensure_ascii=False).encode('utf-8')
        self.send_response(code)
        self.send_header('Content-Type', 'application/json; charset=utf-8')
        self.send_header('Content-Length', str(len(data)))
        if retry_after is not None:
            self.send_header('Retry-After', str(retry_after))
        self.end_headers()
        self.wfile.write(data)

    def do_GET(self):
        path = urlsplit(self.path).path
        if path == '/health':
            with REMOTE_JOB_LOCK:
                running = sum(record.get('status') == 'running' for record in REMOTE_JOBS.values())
            gpu_ready, gpu_message, gpu_devices = remote_gpu_status()
            requested_task = parse_qs(urlsplit(self.path).query).get('task', [REMOTE_ALLOWED_TASKS[0]])[-1]
            inference_ready, inference_message = remote_inference_ready(requested_task)
            if gpu_ready and not inference_ready:
                gpu_message = inference_message
            self.send_json(200, {'ok': True, 'api': 'kaggle-generation-bridge-v3',
                                 'service_id': REMOTE_SERVICE_ID, 'task': requested_task,
                                 'backend_ready': gpu_ready and inference_ready, 'gpu_ready': gpu_ready,
                                 'gpu_message': gpu_message, 'gpu_devices': gpu_devices,
                                 'stalled': remote_stalled(), 'running': running,
                                 'waiting': REMOTE_QUEUE.qsize(), 'queue_capacity': REMOTE_QUEUE.maxsize})
            return
        if path == '/schema':
            self.send_json(200, {'kind': REMOTE_KIND, 'tasks': REMOTE_ALLOWED_TASKS,
                                 'fields': REMOTE_FIELDS, 'image_limit': REMOTE_MAX_IMAGES,
                                 'upload_field': 'images' if REMOTE_MAX_IMAGES else None,
                                 'queue_capacity': REMOTE_QUEUE.maxsize,
                                 'model_switching': 'configure in notebook first'})
            return
        pieces = path.strip('/').split('/')
        if pieces[0] != 'jobs' or len(pieces) not in (2, 3):
            self.send_json(404, {'error': 'Not found'})
            return
        job_id = pieces[1]
        with REMOTE_JOB_LOCK:
            record = REMOTE_JOBS.get(job_id)
            item = dict(record) if record else None
            if item and item['status'] == 'running':
                item['stalled'] = time.monotonic() - item.get('_start_clock', time.monotonic()) > REMOTE_STALL_SECONDS
            if item and item['status'] == 'queued':
                waiting = [key for key, value in REMOTE_JOBS.items() if value.get('status') == 'queued']
                if job_id in waiting:
                    item['queue_position'] = waiting.index(job_id) + 1
            if item and len(pieces) == 3 and pieces[2] in ('file', 'last-frame') and item['status'] == 'done':
                record['_downloads'] = record.get('_downloads', 0) + 1
                downloading = True
            else:
                downloading = False
        if item is None:
            self.send_json(404, {'error': 'Unknown job'})
            return
        if len(pieces) == 2:
            for field in tuple(item):
                if field.startswith('_'):
                    item.pop(field)
            self.send_json(200, item)
            return
        if not downloading:
            self.send_json(404, {'error': 'Result not available for this job'})
            return
        try:
            file_path = item.get('_path') if pieces[2] == 'file' else item.get('_last_frame')
            if not file_path or not Path(file_path).is_file():
                self.send_json(404, {'error': 'Output is no longer available'})
                return
            result = Path(file_path)
            self.send_response(200)
            self.send_header('Content-Type', 'application/octet-stream')
            self.send_header('Content-Disposition', f'attachment; filename="{result.name}"')
            self.send_header('Content-Length', str(result.stat().st_size))
            self.end_headers()
            with result.open('rb') as stream:
                shutil.copyfileobj(stream, self.wfile, length=1024 * 1024)
        except (BrokenPipeError, ConnectionResetError):
            pass
        finally:
            with REMOTE_JOB_LOCK:
                record['_downloads'] -= 1

    def do_POST(self):
        if urlsplit(self.path).path != '/generate':
            self.send_json(404, {'error': 'Not found'})
            return
        supplied_id = self.headers.get('X-Request-ID', '').strip()
        try:
            job_id = uuid.UUID(supplied_id).hex if supplied_id else uuid.uuid4().hex
        except ValueError:
            self.send_json(400, {'error': 'X-Request-ID must be a UUID'})
            return
        with REMOTE_JOB_LOCK:
            existing = REMOTE_JOBS.get(job_id)
        if existing:
            self.send_json(202, {'job_id': job_id, 'status_url': f'/jobs/{job_id}',
                                 'download_url': f'/jobs/{job_id}/file', 'reused': True})
            return
        if remote_stalled():
            self.send_json(503, {'error': 'GPU worker appears stalled; inspect the Notebook before accepting more jobs'}, 60)
            return
        gpu_ready, gpu_message, _ = remote_gpu_status()
        if not gpu_ready:
            self.send_json(409, {'error': gpu_message})
            return
        if shutil.disk_usage(REMOTE_ROOT).free < REMOTE_MIN_FREE_BYTES:
            self.send_json(507, {'error': 'Kaggle disk space is low; clean old outputs before retrying'})
            return
        if not REMOTE_UPLOAD_SLOTS.acquire(timeout=1):
            self.send_json(503, {'error': 'Upload slots busy; retry later'}, 5)
            return
        staging = None
        try:
            self.connection.settimeout(120)
            length = int(self.headers.get('Content-Length', '0'))
            if not 0 < length <= 120 * 1024 * 1024:
                raise ValueError('Request must be between 1 byte and 120 MiB')
            body = self.rfile.read(length)
            if len(body) != length:
                raise ValueError('Incomplete request body')
            content_type = self.headers.get('Content-Type', '')
            fields, files = {}, []
            if content_type.startswith('application/json'):
                fields = json.loads(body)
            elif content_type.startswith('application/x-www-form-urlencoded'):
                fields = {key: values[-1] for key, values in parse_qs(body.decode('utf-8')).items()}
            elif content_type.startswith('multipart/form-data'):
                message = BytesParser(policy=policy.default).parsebytes(
                    b'Content-Type: ' + content_type.encode() + b'\r\nMIME-Version: 1.0\r\n\r\n' + body)
                for part in message.iter_parts():
                    name = part.get_param('name', header='content-disposition')
                    data = part.get_payload(decode=True)
                    if name == 'images':
                        files.append((part.get_filename() or 'upload.png', data))
                    elif name:
                        fields[name] = data.decode('utf-8')
            else:
                raise ValueError('Use application/json, form data, or multipart/form-data')
            if not isinstance(fields, dict):
                raise ValueError('Parameters must be a JSON object')
            if 'abc' in fields and len(str(fields['abc']).encode('utf-8')) > 1024 * 1024:
                raise ValueError('ABC score must be 1 MiB or smaller')
            task = str(fields.get('task', REMOTE_ALLOWED_TASKS[0])).lower()
            if task not in REMOTE_ALLOWED_TASKS:
                raise ValueError('Unsupported task: ' + task)
            if len(files) > REMOTE_MAX_IMAGES:
                raise ValueError(f'Only {REMOTE_MAX_IMAGES} input images are supported')
            if REMOTE_KIND in ('music', 'image_text') and files:
                raise ValueError('This model does not accept reference images')
            if REMOTE_KIND == 'video':
                minimum, maximum = {'t2v': (0, 0), 'i2v': (1, 1), 'flf2v': (2, 2),
                                    'r2v': (1, REMOTE_MAX_IMAGES)}[task] if task == 'r2v' else {
                                    't2v': (0, 0), 'i2v': (1, 1), 'flf2v': (2, 2)}[task]
                if not minimum <= len(files) <= maximum:
                    raise ValueError('Input image count does not match the selected video task')
            if REMOTE_KIND == 'image_multi':
                minimum, maximum = {'text': (0, 0), 'edit': (1, 1), 'multi': (2, REMOTE_MAX_IMAGES)}[task]
                if not minimum <= len(files) <= maximum:
                    raise ValueError(f'Task {task} requires {minimum}–{maximum} images')
            for key in fields:
                if key not in REMOTE_FIELDS:
                    raise ValueError('Unsupported parameter: ' + key)
            remote_validate_fields(task, dict(fields, _image_count=len(files)))
            inference_ready, inference_message = remote_inference_ready(task)
            if not inference_ready:
                self.send_json(409, {'error': inference_message})
                return
            for filename, data in files:
                if not isinstance(data, bytes) or len(data) > 20 * 1024 * 1024:
                    raise ValueError('Each image must be 20 MiB or smaller')
                if Path(filename).suffix.lower() not in {'.png', '.jpg', '.jpeg', '.webp'}:
                    raise ValueError('Only PNG/JPEG/WebP images are supported')
                from PIL import Image
                with Image.open(io.BytesIO(data)) as photo:
                    photo.verify()
            staging = Path(tempfile.mkdtemp(prefix='pending_', dir=REMOTE_ROOT))
            image_names = []
            for index, (filename, data) in enumerate(files, 1):
                target = staging / f'{index:03d}{Path(filename).suffix.lower()}'
                target.write_bytes(data)
                image_names.append(target.name)
            with REMOTE_JOB_LOCK:
                existing = REMOTE_JOBS.get(job_id)
                full = REMOTE_QUEUE.full()
                if not existing and not full:
                    folder = REMOTE_ROOT / job_id
                    staging.rename(folder)
                    staging = None
                    image_paths = [str(folder / name) for name in image_names]
                    REMOTE_JOBS[job_id] = {'id': job_id, 'task': task, 'status': 'queued',
                                           'created_at': time.time()}
                    REMOTE_QUEUE.put_nowait((job_id, fields, image_paths))
                    position = REMOTE_QUEUE.qsize()
            if existing:
                self.send_json(202, {'job_id': job_id, 'status_url': f'/jobs/{job_id}',
                                     'download_url': f'/jobs/{job_id}/file', 'reused': True})
            elif full:
                self.send_json(429, {'error': 'Queue is full; retry later'}, 15)
            else:
                self.send_json(202, {'job_id': job_id, 'status_url': f'/jobs/{job_id}',
                                     'download_url': f'/jobs/{job_id}/file', 'queue_position': position})
        except (ValueError, KeyError, TypeError, json.JSONDecodeError) as error:
            self.send_json(400, {'error': str(error)})
        except (TimeoutError, ConnectionError, OSError) as error:
            self.send_json(503, {'error': f'Upload temporarily failed: {type(error).__name__}'}, 10)
        except Exception as error:
            self.send_json(500, {'error': type(error).__name__ + ': ' + str(error)[:250]})
        finally:
            if staging is not None:
                shutil.rmtree(staging, ignore_errors=True)
            REMOTE_UPLOAD_SLOTS.release()

'''
Convert numeric HTTP form fields into explicitly supported generator settings.
只轉換白名單中的 HTTP 參數，不執行客戶端提供的任何程式碼。
'''
def remote_generate(job_id, fields, image_paths):
    with REMOTE_ENGINE_LOCK:
        with REMOTE_JOB_LOCK:
            REMOTE_JOBS[job_id].update(status='running', _start_clock=time.monotonic())
        namespace = globals()
        keys = ['TASK', 'PROMPT', 'WIDTH', 'HEIGHT', 'SEED', 'LM_SEED', 'SECONDS', 'STEPS', 'CFG',
                'LYRICS', 'TAGS', 'STYLE', 'DURATION', 'GPU_ID', 'COT', 'ABC_PATH',
                'OUTPUT_NAME', 'REFERENCE_PATHS', 'ACTIVE_JOB', 'TASK_CONFIGS',
                'R2V_REF_DIR', 'NEGATIVE_PROMPT', 'SAMPLER', 'REFERENCE_SIZE',
                'OUTPUTS', 'output', 'OUTPUT_PATH', 'OUTPUT_MP4', 'LAST_OUTPUT', 'RUN_SEED']
        before = {key: copy.deepcopy(namespace.get(key)) for key in keys if key in namespace}
        missing = set(keys) - set(before)
        try:
            task = fields.get('task', REMOTE_ALLOWED_TASKS[0]).lower()
            numerics = {'seed': int, 'lm_seed': int, 'width': int, 'height': int, 'steps': int,
                        'seconds': float, 'cfg': float, 'duration': int,
                        'image_strength': float, 'reference_size': int, 'audio_shift': float}
            inputs = {}
            for key, value in fields.items():
                inputs[key] = numerics[key](value) if key in numerics else value

            '''
            Resolve -1 once on the server so the exact value can be reproduced.
            在服務端一次決定 -1 的實際種子，方便重現結果。
            '''
            for seed_key in ('seed', 'lm_seed'):
                if seed_key in inputs:
                    if not -1 <= inputs[seed_key] < 2**32:
                        raise ValueError(f'{seed_key} must be -1 or between 0 and 4294967295')
                    if inputs[seed_key] == -1:
                        inputs[seed_key] = secrets.randbelow(2**32)
            with REMOTE_JOB_LOCK:
                for seed_key in ('seed', 'lm_seed'):
                    if seed_key in inputs:
                        REMOTE_JOBS[job_id][seed_key + '_used'] = inputs[seed_key]
            for dimension in ('width', 'height'):
                if dimension in inputs and not 0 < inputs[dimension] <= 4096:
                    raise ValueError('Width and height must be between 1 and 4096')
            for key in ('seconds', 'duration'):
                if key in inputs and not 0 < inputs[key] <= 3600:
                    raise ValueError('Duration must be between 0 and 3600 seconds')
            if 'reference_size' in inputs and not 64 <= inputs['reference_size'] <= 2048:
                raise ValueError('Reference size must be between 64 and 2048')
            if 'ref_image_size' in inputs and inputs['ref_image_size'] not in ('match', 'max'):
                raise ValueError('Reference image mode must be match or max')
            if 'image_strength' in inputs and not 0 <= inputs['image_strength'] <= 1:
                raise ValueError('Image strength must be between 0 and 1')
            if 'steps' in inputs and not 1 <= inputs['steps'] <= 200:
                raise ValueError('Steps must be between 1 and 200')
            if REMOTE_KIND == 'video':
                if 'TASK_CONFIGS' in namespace:
                    if task not in namespace['TASK_CONFIGS']:
                        raise ValueError('Run this task configuration cell before API generation')
                    updated = copy.deepcopy(namespace['TASK_CONFIGS'][task])
                    for field in ('prompt', 'width', 'height', 'seconds', 'steps', 'seed',
                                  'audio_shift', 'ref_image_size'):
                        if field in inputs:
                            updated[field] = inputs[field]
                    updated['output_name'] = f'remote_{job_id[:12]}.mp4'
                    if task == 'i2v':
                        updated['first_image'] = image_paths[0]
                    elif task == 'flf2v':
                        updated['first_image'], updated['last_image'] = image_paths
                    elif task == 'r2v':
                        updated['ref_images'] = image_paths
                        namespace['R2V_REF_DIR'] = Path(image_paths[0]).parent
                    namespace['TASK_CONFIGS'][task] = updated
                    namespace['TASK'] = task
                else:
                    namespace['ACTIVE_JOB'] = {'mode': task, 'prompt': inputs.get('prompt', namespace.get('PROMPT', '')),
                                               'run_name': f'remote_{job_id[:12]}',
                                               'first_path': image_paths[0] if image_paths else None,
                                               'last_path': image_paths[1] if len(image_paths) > 1 else None}
                    for key in ('width', 'height', 'seconds', 'seed', 'image_strength'):
                        if key in inputs:
                            namespace[key.upper()] = inputs[key]
            else:
                for key in ('prompt', 'width', 'height', 'seed', 'lm_seed', 'steps', 'cfg',
                            'lyrics', 'tags', 'style', 'duration', 'negative_prompt',
                            'sampler', 'reference_size', 'cot'):
                    if key in inputs:
                        namespace[key.upper()] = inputs[key]
                if REMOTE_KIND == 'image_multi':
                    namespace['TASK'] = task
                    namespace['REFERENCE_PATHS'] = image_paths
                if REMOTE_KIND == 'music':
                    namespace['OUTPUT_NAME'] = f'remote_{job_id[:12]}.mp3'
                    if 'ABC_PATH' in namespace:
                        namespace['ABC_PATH'] = ''
                    if 'abc' in inputs:
                        score = (REMOTE_ROOT / job_id / 'input.abc')
                        score.write_text(str(inputs['abc']), encoding='utf-8')
                        namespace['ABC_PATH'] = str(score)
                if REMOTE_KIND in ('image_multi', 'image_text'):
                    namespace['OUTPUT_NAME'] = f'remote_{job_id[:12]}.png'
            for output_key in ('OUTPUTS', 'output', 'OUTPUT_PATH', 'OUTPUT_MP4', 'LAST_OUTPUT'):
                namespace.pop(output_key, None)
            gpu_ready, gpu_message, _ = remote_gpu_status()
            if not gpu_ready:
                raise RuntimeError(gpu_message)
            inference_ready, inference_message = remote_inference_ready(task)
            if not inference_ready:
                raise RuntimeError(inference_message)
            exec(compile(REMOTE_SCRIPT, '<notebook-generation-cell>', 'exec'), namespace)
            paths = []
            if REMOTE_KIND == 'video':
                paths = [namespace.get('LAST_OUTPUT') or namespace.get('OUTPUT_MP4')]
            elif REMOTE_KIND == 'image_text':
                paths = [namespace.get('OUTPUT_PATH')]
            elif REMOTE_KIND == 'image_multi':
                paths = [namespace.get('output')]
            elif REMOTE_KIND == 'music':
                paths = namespace.get('OUTPUTS', []) if 'OUTPUTS' in namespace else [namespace.get('output')]
            results = [Path(path) for path in paths if path is not None]
            if REMOTE_KIND == 'music' and REMOTE_SERVICE_ID.startswith('06_YuE2_'):
                results += [sidecar for path in list(results)
                            for ext in ('.json', '.abc')
                            if (sidecar := path.with_suffix(ext)).is_file()]
            if not results or any(not path.is_file() for path in results):
                raise FileNotFoundError('Generation finished without a saved result file')
            folder = REMOTE_ROOT / job_id
            if len(results) == 1:
                target = folder / f'result{results[0].suffix.lower()}'
                shutil.copy2(results[0], target)
            else:
                target = folder / 'results.zip'
                with zipfile.ZipFile(target, 'w', compression=zipfile.ZIP_STORED) as archive:
                    for result in results:
                        archive.write(result, arcname=result.name)
            last_frame = None
            if REMOTE_KIND == 'video' and shutil.which('ffprobe') and shutil.which('ffmpeg'):
                try:
                    probe = subprocess.run(['ffprobe', '-v', 'error', '-count_frames',
                                            '-select_streams', 'v:0', '-show_entries',
                                            'stream=nb_read_frames', '-of', 'json', str(target)],
                                           check=True, text=True, capture_output=True, timeout=120)
                    index = int(json.loads(probe.stdout)['streams'][0]['nb_read_frames']) - 1
                    if index >= 0:
                        last_frame = folder / 'last_frame.png'
                        subprocess.run(['ffmpeg', '-y', '-loglevel', 'error', '-i', str(target),
                                        '-vf', rf'select=eq(n\,{index})', '-frames:v', '1',
                                        str(last_frame)], check=True)
                except (OSError, ValueError, KeyError, subprocess.CalledProcessError, subprocess.TimeoutExpired):
                    last_frame = None

            '''
            Remove only source files bearing this remote job's unique prefix.
            僅清理本次遠端工作專屬原始輸出，保留 Notebook 手動生成的檔案。
            '''
            for original in results:
                if original.name.startswith(f'remote_{job_id[:12]}') and original != target:
                    try:
                        original.unlink(missing_ok=True)
                    except OSError:
                        pass
            comfy = globals().get('COMFY')
            if comfy is not None and (Path(comfy) / 'output').exists():
                for artifact in (Path(comfy) / 'output').rglob(f'remote_{job_id[:12]}*'):
                    if artifact.is_file():
                        try:
                            artifact.unlink()
                        except OSError:
                            pass
            with REMOTE_JOB_LOCK:
                REMOTE_JOBS[job_id].update(status='done', filename=target.name,
                                           bytes=target.stat().st_size, _path=str(target),
                                           last_frame_available=bool(last_frame),
                                           _last_frame=str(last_frame) if last_frame else None)
        except Exception as error:
            with REMOTE_JOB_LOCK:
                REMOTE_JOBS[job_id].update(status='error', error=f'{type(error).__name__}: {error}'[:2000])
        finally:
            for key in missing:
                namespace.pop(key, None)
            namespace.update(before)

'''
Use one GPU worker, regardless of the number of HTTP request threads.
不論有多少 HTTP 連線，都只有一個 GPU worker 消費佇列。
'''
def remote_worker():
    while True:
        job_id, fields, image_paths = REMOTE_QUEUE.get()
        try:
            remote_generate(job_id, fields, image_paths)
        except Exception as error:
            with REMOTE_JOB_LOCK:
                if job_id in REMOTE_JOBS:
                    REMOTE_JOBS[job_id].update(status='error', error=f'{type(error).__name__}: {error}'[:2000])
        finally:
            with REMOTE_JOB_LOCK:
                if job_id in REMOTE_JOBS:
                    REMOTE_JOBS[job_id]['finished_at'] = time.time()
            try:
                remote_remove_inputs(job_id)
            except OSError as error:
                print('API input cleanup warning:', str(error)[:180])
            finally:
                REMOTE_QUEUE.task_done()

if not (globals().get('REMOTE_WORKER') and REMOTE_WORKER.is_alive()):
    REMOTE_WORKER = threading.Thread(target=remote_worker, daemon=True)
    REMOTE_WORKER.start()

if not (globals().get('REMOTE_MAINTENANCE') and REMOTE_MAINTENANCE.is_alive()):
    REMOTE_MAINTENANCE = threading.Thread(target=remote_maintenance, daemon=True)
    REMOTE_MAINTENANCE.start()

old_tunnel = globals().get('REMOTE_NGROK_PROC')
if old_tunnel is not None and old_tunnel.poll() is None:
    old_tunnel.terminate()
    try:
        old_tunnel.wait(timeout=5)
    except subprocess.TimeoutExpired:
        old_tunnel.kill()
        old_tunnel.wait(timeout=5)

NGROK_URL = ''

'''
Install the official ngrok agent only if no usable binary exists.
系統尚無 ngrok 才下載官方執行檔，不重複安裝。
'''
ngrok_bin = shutil.which('ngrok') or str(ROOT / 'ngrok')
if not Path(ngrok_bin).is_file():
    url = 'https://bin.ngrok.com/c/bNyj1mQVY4c/ngrok-v3-stable-linux-amd64.tgz'
    with urllib.request.urlopen(url, timeout=90) as response:
        blob = response.read()
    with tarfile.open(fileobj=io.BytesIO(blob), mode='r:gz') as archive:
        entry = next((member for member in archive.getmembers() if Path(member.name).name == 'ngrok' and member.isfile()), None)
        if entry is None:
            raise RuntimeError('ngrok 官方下載檔案缺少執行檔')
        Path(ngrok_bin).write_bytes(archive.extractfile(entry).read())
    Path(ngrok_bin).chmod(0o755)

ngrok_token = getpass('輸入 ngrok Authtoken：').strip()
if not ngrok_token:
    raise ValueError('未輸入 ngrok Authtoken')

result = subprocess.run([ngrok_bin, 'config', 'add-authtoken', ngrok_token], capture_output=True, text=True)
if result.returncode:
    raise RuntimeError('ngrok token 設定失敗（請確認 Token 是否有效）')

del ngrok_token
result = subprocess.run([ngrok_bin, 'config', 'check'], capture_output=True, text=True)
if result.returncode:
    raise RuntimeError('ngrok 設定格式檢查失敗：' + result.stderr[-500:])

server = globals().get('REMOTE_SERVER')
server_thread = globals().get('REMOTE_SERVER_THREAD')
server_is_current = (
    server is not None
    and server_thread is not None
    and server_thread.is_alive()
    and getattr(server, 'notebook_id', None) == REMOTE_SERVICE_ID
)
if server is not None and not server_is_current:
    if server_thread is not None and server_thread.is_alive():
        server.shutdown()
    server.server_close()

if not server_is_current:
    try:
        REMOTE_SERVER = ThreadingHTTPServer(('127.0.0.1', REMOTE_PORT), RemoteHandler)
    except OSError as error:
        raise RuntimeError(f'本機 API Port {REMOTE_PORT} 已被其他服務占用') from error
    REMOTE_SERVER.notebook_id = REMOTE_SERVICE_ID
    REMOTE_SERVER.daemon_threads = True
    REMOTE_SERVER_THREAD = threading.Thread(target=REMOTE_SERVER.serve_forever, daemon=True)
    REMOTE_SERVER_THREAD.start()
    print('已啟動新的本機 API Server。')
else:
    REMOTE_SERVER.RequestHandlerClass = RemoteHandler
    print('已沿用本機 API Server；現有生成工作與結果保持不變。')

with urllib.request.urlopen(f'http://127.0.0.1:{REMOTE_PORT}/health', timeout=4) as response:
    if json.load(response).get('ok') is not True:
        raise RuntimeError('本機 API Server 尚未就緒')

'''
Use the agent inspection API to obtain the actual public URL.
透過 ngrok 本機檢查 API 取得實際對外 HTTPS 網址。
'''
NGROK_LOG = REMOTE_ROOT / 'ngrok.log'
with NGROK_LOG.open('w') as log:
    REMOTE_NGROK_PROC = subprocess.Popen([ngrok_bin, 'http', str(REMOTE_PORT)], stdout=log,
                                         stderr=subprocess.STDOUT, start_new_session=True)

NGROK_URL = ''
for attempt in range(25):
    if REMOTE_NGROK_PROC.poll() is not None:
        break
    try:
        with urllib.request.urlopen('http://127.0.0.1:4040/api/tunnels', timeout=2) as response:
            tunnels = json.load(response).get('tunnels', [])
        NGROK_URL = next((item['public_url'] for item in tunnels
                          if item.get('public_url', '').startswith('https://') and
                          str(REMOTE_PORT) in item.get('config', {}).get('addr', '')), '')
        if NGROK_URL:
            break
    except (OSError, ValueError, KeyError):
        pass
    time.sleep(1)

if not NGROK_URL:
    raise RuntimeError('ngrok URL 建立失敗：\n' + NGROK_LOG.read_text(errors='replace')[-2500:])

print('ngrok HTTPS URL:', NGROK_URL)
print('提醒：公開的 ngrok API URL 不需要額外驗證，請只分享給授課學生。')
print('GET /health | GET /schema | POST /generate | GET /jobs/{id} | GET /jobs/{id}/file')
print('影片輸出另提供：GET /jobs/{id}/last-frame')
print('✓ 第 7 節完成：選用 FIFO／ngrok API 已啟動。')


8. （選用）遠端 API 與 client 使用說明


In [ ]:
'''
Query the active bridge for supported parameters and task names.
從正在執行的橋接服務讀取實際支援的參數與模式。
'''
import json
import urllib.request
request = urllib.request.Request(f'http://127.0.0.1:{REMOTE_PORT}/schema')
with urllib.request.urlopen(request, timeout=5) as response:
    schema = json.load(response)

print(json.dumps(schema, ensure_ascii=False, indent=2))
print('用法：POST NGROK_URL + "/generate"（JSON 或 multipart/images），取得 job_id；')
print('輪詢 GET /jobs/{job_id}；完成後下載 GET /jobs/{job_id}/file。')
print('本機 requests 範例：06_Z-Image-Turbo_圖片生成_client.py（與本 Notebook 同名對應）。')
print('使用前只需在對應的 client.py 填入 API_URL。')
print('本服務使用單一 FIFO worker，遠端請求不會同時啟動多筆 GPU 推論。')
print('✓ 第 8 節完成：API Schema 和 client 說明已顯示。')
